In [1]:
import os
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from dotenv import load_dotenv
from supabase import create_client, Client
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report

c:\DATA REVEL\NgodingMoment\IKD-PreThesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
URL = os.getenv("SUPABASE_URL")
KEY = os.getenv("SUPABASE_KEY")
BUCKET_NAME = os.getenv("SUPABASE_BUCKET")
supabase: Client = create_client(URL, KEY)

In [3]:
cloud_path = f"{URL}/storage/v1/object/public/{BUCKET_NAME}/ulasan_ikd_preprocessed.csv"

df = pd.read_csv(cloud_path)

# [KOREKSI FATAL]: Kembalikan NaN menjadi string kosong khusus untuk kolom ablasi
df = df.fillna("")

print("Data loaded successfully. Shape:", df.shape)

X = df['text_baseline']
y = df['label']

Data loaded successfully. Shape: (34902, 7)


In [4]:
# 2. Data Splitting (Train: 64%, Val: 16%, Test: 20%)
# Train & Val digunakan untuk Optuna. Test disimpan murni untuk evaluasi akhir.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Data Train+Val (5-Fold CV): {len(X_temp)} baris | Blind Test Set: {len(X_test)} baris")

Data Train+Val (5-Fold CV): 27921 baris | Blind Test Set: 6981 baris


In [5]:
# optuna.delete_study(study_name="SVM_IKD_Tuning_v4", storage="sqlite:///svm_tuning.sqlite3")

In [6]:
db_file = "svm_tuning.sqlite3"
storage_url = f"sqlite:///{db_file}"

def tune_svm(kernel_type: str, n_trials: int):
    def objective(trial):
        ngram_name = trial.suggest_categorical("ngram_range", ["unigram", "bigram"])
        ngram_param = (1, 1) if ngram_name == "unigram" else (1, 2)
        min_df_param = trial.suggest_int("min_df", 1, 5)
        max_df_param = trial.suggest_float("max_df", 0.7, 1.0)
        c_param = trial.suggest_float("C", 0.1, 10.0, log=True)

        if kernel_type == "linear":
            clf = LinearSVC(C=c_param, random_state=42, dual="auto")
        else:
            gamma_param = trial.suggest_categorical("gamma", ["scale", "auto"])
            clf = SVC(C=c_param, kernel="rbf", gamma=gamma_param, random_state=42)

        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=ngram_param,
                min_df=min_df_param,
                max_df=max_df_param
            )),
            ("svm", clf)
        ])

        scores = cross_val_score(
            pipe, X_temp, y_temp, cv=cv, scoring="f1_macro", n_jobs=-1
        )
        return scores.mean()

    study = optuna.create_study(
        direction="maximize",
        study_name=f"SVM_IKD_{kernel_type}_5Fold_v1",
        storage=storage_url,
        sampler=TPESampler(seed=42),
        load_if_exists=True
    )
    study.optimize(objective, n_trials=n_trials)
    return study

print("1. Tuning Linear SVM (30 trials, 5-Fold CV)...")
study_linear = tune_svm("linear", n_trials=30)

print("\n2. Tuning RBF SVM (25 trials, 5-Fold CV)...")
study_rbf = tune_svm("rbf", n_trials=25)

for name, st in [("LINEAR", study_linear), ("RBF", study_rbf)]:
    print(f"\n=== BEST CV RESULT ({name}) ===")
    print(f"Mean 5-Fold Macro F1: {st.best_value:.4f}")
    print("Best Params:", st.best_params)

1. Tuning Linear SVM (30 trials, 5-Fold CV)...


[I 2026-11-19 11:29:32,871] A new study created in RDB with name: SVM_IKD_linear_5Fold_v1
[I 2026-11-19 11:29:39,430] Trial 0 finished with value: 0.9272099239230208 and parameters: {'ngram_range': 'bigram', 'min_df': 4, 'max_df': 0.8795975452591109, 'C': 0.20513382630874505}. Best is trial 0 with value: 0.9272099239230208.
[I 2026-11-19 11:29:44,058] Trial 1 finished with value: 0.9170871046817423 and parameters: {'ngram_range': 'unigram', 'min_df': 5, 'max_df': 0.8803345035229626, 'C': 2.607024758370768}. Best is trial 0 with value: 0.9272099239230208.
[I 2026-11-19 11:29:49,071] Trial 2 finished with value: 0.9273137296573106 and parameters: {'ngram_range': 'bigram', 'min_df': 5, 'max_df': 0.7637017332034828, 'C': 0.23102018878452935}. Best is trial 2 with value: 0.9273137296573106.
[I 2026-11-19 11:29:54,691] Trial 3 finished with value: 0.9265119292711356 and parameters: {'ngram_range': 'bigram', 'min_df': 3, 'max_df': 0.8295835055926347, 'C': 0.38234752246751863}. Best is trial 2


2. Tuning RBF SVM (25 trials, 5-Fold CV)...


[I 2026-11-19 11:32:07,141] Trial 0 finished with value: 0.9105970754487543 and parameters: {'ngram_range': 'bigram', 'min_df': 4, 'max_df': 0.8795975452591109, 'C': 0.20513382630874505, 'gamma': 'scale'}. Best is trial 0 with value: 0.9105970754487543.
[I 2026-11-19 11:35:08,108] Trial 1 finished with value: 0.9227222477637248 and parameters: {'ngram_range': 'unigram', 'min_df': 4, 'max_df': 0.7061753482887407, 'C': 8.706020878304859, 'gamma': 'scale'}. Best is trial 1 with value: 0.9227222477637248.
[I 2026-11-19 11:37:31,512] Trial 2 finished with value: 0.3714034835239042 and parameters: {'ngram_range': 'bigram', 'min_df': 2, 'max_df': 0.8574269294896714, 'C': 0.7309539835912913, 'gamma': 'auto'}. Best is trial 1 with value: 0.9227222477637248.
[I 2026-11-19 11:38:49,108] Trial 3 finished with value: 0.3714034835239042 and parameters: {'ngram_range': 'bigram', 'min_df': 2, 'max_df': 0.8368209952651108, 'C': 3.7183641805732095, 'gamma': 'auto'}. Best is trial 1 with value: 0.9227222


=== BEST CV RESULT (LINEAR) ===
Mean 5-Fold Macro F1: 0.9280
Best Params: {'ngram_range': 'bigram', 'min_df': 5, 'max_df': 0.9180131945740828, 'C': 0.17620634878958852}

=== BEST CV RESULT (RBF) ===
Mean 5-Fold Macro F1: 0.9270
Best Params: {'ngram_range': 'unigram', 'min_df': 3, 'max_df': 0.7148964401981572, 'C': 1.3960604996502022, 'gamma': 'scale'}


```
=== HASIL TERBAIK OPTUNA - Trial 16 ===
Best Macro F1 (Validation): 0.9262
Best Parameters:
  ngram_range: unigram
  min_df: 4
  max_df: 0.7960612188090677
  C: 2.0348244290341504
  kernel: rbf
  gamma: scale
```

In [7]:
def build_best_pipeline(best_params: dict, kernel_type: str) -> Pipeline:
    ngram_param = (1, 1) if best_params["ngram_range"] == "unigram" else (1, 2)
    if kernel_type == "linear":
        clf = LinearSVC(C=best_params["C"], random_state=42, dual="auto")
    else:
        clf = SVC(
            C=best_params["C"],
            kernel="rbf",
            gamma=best_params["gamma"],
            random_state=42
        )
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=ngram_param,
            min_df=best_params["min_df"],
            max_df=best_params["max_df"]
        )),
        ("svm", clf)
    ])

# Hitung baseline Default Linear SVM pada Blind Test Set secara terprogram (bukan hardcode)
default_pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svm", SVC(kernel="linear", random_state=42))
])
default_pipe.fit(X_temp, y_temp)
f1_default_test = f1_score(y_test, default_pipe.predict(X_test), average="macro")

results = {"Default Linear SVM": f1_default_test}

for label, st, k_type in [
    ("Tuned Linear SVM", study_linear, "linear"),
    ("Tuned RBF SVM", study_rbf, "rbf")
]:
    best_pipe = build_best_pipeline(st.best_params, k_type)
    best_pipe.fit(X_temp, y_temp)
    y_pred_test = best_pipe.predict(X_test)
    results[label] = f1_score(y_test, y_pred_test, average="macro")

    print(f"\n=== CLASSIFICATION REPORT: {label.upper()} (BLIND TEST) ===")
    print(classification_report(y_test, y_pred_test, digits=4))

print("\n=== RINGKASAN PERBANDINGAN FINAL (BLIND TEST MACRO F1) ===")
for model_name, score in results.items():
    print(f"{model_name:<22}: {score:.4f}")


=== CLASSIFICATION REPORT: TUNED LINEAR SVM (BLIND TEST) ===
              precision    recall  f1-score   support

           0     0.9240    0.9673    0.9452      4125
           1     0.9493    0.8852    0.9161      2856

    accuracy                         0.9337      6981
   macro avg     0.9367    0.9262    0.9306      6981
weighted avg     0.9344    0.9337    0.9333      6981


=== CLASSIFICATION REPORT: TUNED RBF SVM (BLIND TEST) ===
              precision    recall  f1-score   support

           0     0.9238    0.9675    0.9452      4125
           1     0.9496    0.8848    0.9161      2856

    accuracy                         0.9337      6981
   macro avg     0.9367    0.9262    0.9306      6981
weighted avg     0.9344    0.9337    0.9333      6981


=== RINGKASAN PERBANDINGAN FINAL (BLIND TEST MACRO F1) ===
Default Linear SVM    : 0.9265
Tuned Linear SVM      : 0.9306
Tuned RBF SVM         : 0.9306


```
4. Melatih ulang model terbaik pada seluruh data training & evaluasi di Data Test (Data Buta)...

=== CLASSIFICATION REPORT (DATA TEST) ===
              precision    recall  f1-score   support

           0     0.9176    0.9665    0.9414      4125
           1     0.9476    0.8746    0.9097      2856

    accuracy                         0.9290      6981
   macro avg     0.9326    0.9206    0.9256      6981
weighted avg     0.9299    0.9290    0.9284      6981

Skor Uji Ablasi (Default) : 0.9269
Skor Final Optuna (Tuned) : 0.9256
```

In [8]:
print(f"Mengunggah riwayat eksperimen ({db_file}) ke Supabase...")
with open(db_file, 'rb') as f:
    supabase.storage.from_(BUCKET_NAME).upload(
        file=f, 
        path=db_file, 
        file_options={"x-upsert": "true"}
    )
print("✅ Riwayat Tuning SVM berhasil diamankan ke Supabase. Tim dapat mengaksesnya sekarang.")

Mengunggah riwayat eksperimen (svm_tuning.sqlite3) ke Supabase...
✅ Riwayat Tuning SVM berhasil diamankan ke Supabase. Tim dapat mengaksesnya sekarang.
